#  This is the bronze layer where we gather the data.  

In our case we have pre-made datasets, these datasets where made by AI.  
The datasets that where created are in blob-format, with the ID, Title, and content sepperated.  


In [33]:
from __future__ import annotations

import json
import logging
import os
import shutil
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from pypdf import PdfReader

# Optional OCR fallback dependencies.
# The pipeline still works without them; OCR is only used when installed.
try:
    import pytesseract  # type: ignore
    from pdf2image import convert_from_path  # type: ignore

    OCR_AVAILABLE = True
except Exception:
    OCR_AVAILABLE = False


In [34]:
RAW_FOLDER = Path("../../Data/raw")
BRONZE_FOLDER = Path("../../Data/bronze")
LOGS_FOLDER = BRONZE_FOLDER / "logs"
REPORTS_FOLDER = BRONZE_FOLDER / "reports"
VALIDATION_REPORT_PATH = REPORTS_FOLDER / "bronze_validation_report.json"
MANIFEST_PATH = REPORTS_FOLDER / "bronze_manifest.json"


In [35]:
@dataclass
class PageExtractionLog:
    page_number: int
    extraction_method: str
    characters_extracted: int
    status: str
    warning: str | None = None


@dataclass
class DocumentProcessingResult:
    document_id: str
    source_file_name: str
    source_file_path: str
    output_json_path: str
    output_txt_path: str
    total_pages: int
    extracted_characters: int
    extraction_status: str
    used_ocr_fallback: bool
    pdf_metadata: dict[str, Any]
    processed_at_utc: str
    error_message: str | None = None


@dataclass
class ValidationSummary:
    total_input_pdfs: int
    successful_documents: int
    failed_documents: int
    empty_documents: int
    partial_documents: int
    ocr_documents: int
    generated_at_utc: str
    documents: list[dict[str, Any]]


logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("bronze-layer")

In [36]:
def clean_raw_txt_files(folder: Path) -> None:
    folder.mkdir(parents=True, exist_ok=True)
    for file_path in folder.iterdir():
        if file_path.is_file() and file_path.name.startswith("doc_") and file_path.suffix.lower() == ".txt":
            file_path.unlink()



def clean_bronze_folder(folder: Path) -> None:
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)
    LOGS_FOLDER.mkdir(parents=True, exist_ok=True)
    REPORTS_FOLDER.mkdir(parents=True, exist_ok=True)



def get_next_doc_id(counter: int) -> str:
    return f"doc_{counter:02d}"



def safe_pdf_metadata(reader: PdfReader) -> dict[str, Any]:
    metadata = reader.metadata or {}
    cleaned: dict[str, Any] = {}

    for key, value in metadata.items():
        normalized_key = str(key).lstrip("/")
        if value is None:
            cleaned[normalized_key] = None
        else:
            cleaned[normalized_key] = str(value)

    return cleaned

In [37]:

def extract_text_with_pypdf(pdf_path: Path) -> tuple[str, list[PageExtractionLog], int]:
    reader = PdfReader(str(pdf_path), strict=False)
    page_logs: list[PageExtractionLog] = []
    page_texts: list[str] = []
    empty_pages = 0

    for index, page in enumerate(reader.pages, start=1):
        extracted = page.extract_text() or ""
        extracted = extracted.strip()

        if extracted:
            page_texts.append(extracted)
            page_logs.append(
                PageExtractionLog(
                    page_number=index,
                    extraction_method="pypdf",
                    characters_extracted=len(extracted),
                    status="success",
                )
            )
        else:
            empty_pages += 1
            page_logs.append(
                PageExtractionLog(
                    page_number=index,
                    extraction_method="pypdf",
                    characters_extracted=0,
                    status="empty",
                    warning="No embedded text detected on this page.",
                )
            )

    return "\n\n".join(page_texts).strip(), page_logs, empty_pages


In [38]:
def extract_text_with_ocr(pdf_path: Path, page_logs: list[PageExtractionLog]) -> tuple[str, list[PageExtractionLog]]:
    if not OCR_AVAILABLE:
        return "", page_logs

    images = convert_from_path(str(pdf_path))
    ocr_texts: list[str] = []
    updated_logs: list[PageExtractionLog] = []

    for image, original_log in zip(images, page_logs, strict=False):
        if original_log.status == "success":
            updated_logs.append(original_log)
            continue

        ocr_text = pytesseract.image_to_string(image).strip()
        if ocr_text:
            ocr_texts.append(ocr_text)
            updated_logs.append(
                PageExtractionLog(
                    page_number=original_log.page_number,
                    extraction_method="ocr",
                    characters_extracted=len(ocr_text),
                    status="success",
                    warning="Recovered text using OCR fallback.",
                )
            )
        else:
            updated_logs.append(
                PageExtractionLog(
                    page_number=original_log.page_number,
                    extraction_method="ocr",
                    characters_extracted=0,
                    status="failed",
                    warning="OCR fallback produced no text.",
                )
            )

    return "\n\n".join(ocr_texts).strip(), updated_logs



def write_text_file(txt_path: Path, text: str) -> None:
    txt_path.write_text(text, encoding="utf-8")



def write_json(path: Path, payload: Any) -> None:
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")



def build_bronze_record(
    document_id: str,
    pdf_path: Path,
    raw_text: str,
    pdf_metadata: dict[str, Any],
    page_logs: list[PageExtractionLog],
    extraction_status: str,
    used_ocr_fallback: bool,
) -> dict[str, Any]:
    return {
        "document_id": document_id,
        "source_file_name": pdf_path.name,
        "source_file_path": str(pdf_path.resolve()),
        "raw_text": raw_text,
        "pdf_metadata": pdf_metadata,
        "page_count": len(page_logs),
        "page_extraction_log": [asdict(log) for log in page_logs],
        "extraction_status": extraction_status,
        "used_ocr_fallback": used_ocr_fallback,
        "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    }

In [39]:
def process_pdf(pdf_path: Path, document_id: str) -> DocumentProcessingResult:
    txt_path = RAW_FOLDER / f"{document_id}.txt"
    bronze_output_path = BRONZE_FOLDER / f"{document_id}_bronze.json"
    page_log_path = LOGS_FOLDER / f"{document_id}_page_log.json"

    reader = PdfReader(str(pdf_path), strict=False)
    pdf_metadata = safe_pdf_metadata(reader)

    extracted_text = ""
    page_logs: list[PageExtractionLog] = []
    used_ocr_fallback = False

    for index, page in enumerate(reader.pages, start=1):
        try:
            extracted = (page.extract_text() or "").strip()
            if extracted:
                extracted_text += extracted + "\n\n"
                page_logs.append(
                    PageExtractionLog(
                        page_number=index,
                        extraction_method="pypdf",
                        characters_extracted=len(extracted),
                        status="success",
                    )
                )
            else:
                page_logs.append(
                    PageExtractionLog(
                        page_number=index,
                        extraction_method="pypdf",
                        characters_extracted=0,
                        status="empty",
                        warning="No embedded text detected on this page.",
                    )
                )
        except Exception as exc:
            page_logs.append(
                PageExtractionLog(
                    page_number=index,
                    extraction_method="pypdf",
                    characters_extracted=0,
                    status="failed",
                    warning=str(exc),
                )
            )

    extracted_text = extracted_text.strip()

    successful_pages = sum(1 for log in page_logs if log.status == "success")
    failed_pages = sum(1 for log in page_logs if log.status == "failed")
    empty_pages = sum(1 for log in page_logs if log.status == "empty")

    if successful_pages == 0 and failed_pages > 0:
        extraction_status = "failed"
    elif successful_pages == 0 and empty_pages == len(page_logs):
        extraction_status = "empty"
    elif failed_pages > 0 or empty_pages > 0:
        extraction_status = "partial"
    else:
        extraction_status = "success"

    write_text_file(txt_path, extracted_text)
    write_json(page_log_path, [asdict(log) for log in page_logs])

    bronze_record = build_bronze_record(
        document_id=document_id,
        pdf_path=pdf_path,
        raw_text=extracted_text,
        pdf_metadata=pdf_metadata,
        page_logs=page_logs,
        extraction_status=extraction_status,
        used_ocr_fallback=used_ocr_fallback,
    )
    write_json(bronze_output_path, [bronze_record])

    return DocumentProcessingResult(
        document_id=document_id,
        source_file_name=pdf_path.name,
        source_file_path=str(pdf_path.resolve()),
        output_json_path=str(bronze_output_path.resolve()),
        output_txt_path=str(txt_path.resolve()),
        total_pages=len(page_logs),
        extracted_characters=len(extracted_text),
        extraction_status=extraction_status,
        used_ocr_fallback=used_ocr_fallback,
        pdf_metadata=pdf_metadata,
        processed_at_utc=datetime.now(timezone.utc).isoformat(),
        error_message=None,
    )
    
    def build_validation_summary(results: list[DocumentProcessingResult], total_input_pdfs: int) -> ValidationSummary:
        successful_documents = sum(1 for result in results if result.extraction_status == "success")
        failed_documents = sum(1 for result in results if result.extraction_status == "failed")
        empty_documents = sum(1 for result in results if result.extraction_status == "empty")
        partial_documents = sum(1 for result in results if result.extraction_status == "partial")
        ocr_documents = sum(1 for result in results if result.used_ocr_fallback)

        return ValidationSummary(
            total_input_pdfs=total_input_pdfs,
            successful_documents=successful_documents,
            failed_documents=failed_documents,
            empty_documents=empty_documents,
            partial_documents=partial_documents,
            ocr_documents=ocr_documents,
            generated_at_utc=datetime.now(timezone.utc).isoformat(),
            documents=[asdict(result) for result in results],
        )

In [40]:
def main() -> None:
    clean_raw_txt_files(RAW_FOLDER)
    clean_bronze_folder(BRONZE_FOLDER)

    pdf_files = sorted(path for path in RAW_FOLDER.iterdir() if path.is_file() and path.suffix.lower() == ".pdf")
    results: list[DocumentProcessingResult] = []
    counter = 1

    if not pdf_files:
        logger.warning("No PDF files found in %s", RAW_FOLDER)

    for pdf_path in pdf_files:
        document_id = get_next_doc_id(counter)
        logger.info("Processing %s -> %s", pdf_path.name, document_id)

        try:
            result = process_pdf(pdf_path, document_id)
            results.append(result)
            logger.info(
                "Created %s | status=%s | chars=%s | ocr=%s",
                result.output_json_path,
                result.extraction_status,
                result.extracted_characters,
                result.used_ocr_fallback,
            )
            counter += 1
        except Exception as exc:
            logger.exception("Failed to process %s: %s", pdf_path.name, exc)
            results.append(
                DocumentProcessingResult(
                    document_id=document_id,
                    source_file_name=pdf_path.name,
                    source_file_path=str(pdf_path.resolve()),
                    output_json_path=str((BRONZE_FOLDER / f"{document_id}_bronze.json").resolve()),
                    output_txt_path=str((RAW_FOLDER / f"{document_id}.txt").resolve()),
                    total_pages=0,
                    extracted_characters=0,
                    extraction_status="failed",
                    used_ocr_fallback=False,
                    pdf_metadata={},
                    processed_at_utc=datetime.now(timezone.utc).isoformat(),
                    error_message=str(exc),
                )
            )
            counter += 1

    summary = build_validation_summary(results, total_input_pdfs=len(pdf_files))
    write_json(VALIDATION_REPORT_PATH, asdict(summary))
    write_json(MANIFEST_PATH, summary.documents)

    logger.info("Validation report written to %s", VALIDATION_REPORT_PATH)
    logger.info("Manifest written to %s", MANIFEST_PATH)
    logger.info(
        "Run complete | total=%s success=%s partial=%s empty=%s failed=%s ocr=%s",
        summary.total_input_pdfs,
        summary.successful_documents,
        summary.partial_documents,
        summary.empty_documents,
        summary.failed_documents,
        summary.ocr_documents,
    )


if __name__ == "__main__":
    main()


INFO | Processing ZS-zeeuwse-arbeidsmarkt-uitgave-1.pdf -> doc_01
INFO | Created D:\Data\School\HZ\year 3\Courses\Internship\BERT-testing\Data\bronze\doc_01_bronze.json | status=success | chars=44267 | ocr=False
INFO | Validation report written to ..\..\Data\bronze\reports\bronze_validation_report.json
INFO | Manifest written to ..\..\Data\bronze\reports\bronze_manifest.json
INFO | Run complete | total=1 success=1 partial=0 empty=0 failed=0 ocr=0
